# 摄像头巡线小车

## 项目简介

本项目实现一个由摄像头侧负责图像巡线、Arduino 侧负责底盘执行的摄像头巡线小车。OpenMV、树莓派或其它视觉模块负责采集道路图像、提取中心线并计算横向误差，再通过串口把误差、路口标记和停车标志发送给 Arduino，Arduino 根据误差进行差速控制。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| OpenMV / 树莓派 / 视觉模块 | 1 | 图像处理与误差计算 |
| 摄像头 | 1 | 路面图像采集 |
| Arduino Uno | 1 | 执行底盘控制 |
| 双路电机驱动 | 1 | 控制左右轮 |
| 直流减速电机 | 2 | 底盘驱动 |


## 核心功能

- 摄像头采集道路图像并计算中心线偏差。
- 视觉端通过串口下发偏差值和特殊路口标记。
- Arduino 使用 PD 差速控制修正方向。
- 摄像头掉帧或串口超时后自动停车保护。
- 具备路口计数和目标路口停车逻辑。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| 左电机 PWM / 方向 | D5、D6、D7 |
| 右电机 PWM / 方向 | D9、D10、D11 |
| 视觉模块串口通信 | USB Serial 或 TTL UART |


## 接线说明

- 左右电机通过驱动模块连接到底盘，PWM 用于控速，方向脚用于前进方向控制。
- 若视觉模块通过 USB 接入 Arduino，则可直接使用 USB Serial；若通过 TTL UART，需要交叉连接 TX/RX 并共地。
- 视觉模块与 Arduino 之间必须约定统一的串口协议和波特率。
- 底盘首次调试时建议架空轮子，先验证误差正负与转向方向是否一致。


## 串口协议

- 视觉帧格式：`L,error,cross,stop`。
- 示例：`L,-18,0,0` 表示当前中心线偏左，需要向左修正；`cross=0` 表示不是路口，`stop=0` 表示继续前进。
- `cross=1` 用于表示检测到路口或任务点，Arduino 侧会做上升沿计数。
- `stop=1` 用于表示检测到终点、红灯或需要人工接管，Arduino 会立即停车。


## 运行方式

1. 打开 `src/camera_line_following_car/camera_line_following_car.ino` 并上传到 Arduino。
2. 在 OpenMV 或树莓派端编写图像巡线程序，实时计算中心线误差。
3. 按协议持续发送误差值、路口标记和停车标记。
4. 根据底盘转向灵敏度调整 `KP`、`KD` 与 `BASE_SPEED`。
5. 根据任务路线调整 `TARGET_INTERSECTION_COUNT`，使小车在目标路口停下。


## 控制逻辑说明

- `parseVisionFrame()` 解析视觉端发来的误差、路口标志和停车标志。
- 当 `crossFlag` 从 0 跳到 1 时，Arduino 才增加一次路口计数，避免同一路口多帧重复累加。
- `applySteering()` 使用误差和误差变化量形成简单 PD 修正，输出左右轮差速。
- 若超过 `FRAME_TIMEOUT_MS` 没有收到新视觉帧，系统立即停车，避免带着旧误差继续运动。
- 当前版本默认保持前进，通过速度差修正方向，适合基础摄像头巡线原型验证。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 误差传输 | Arduino 能连续接收视觉误差帧 |
| 巡线控制 | 小车能根据误差完成左右修正 |
| 路口计数 | 经过目标路口时计数正确增加 |
| 超时保护 | 停止发送视觉帧后小车自动停车 |


## 可扩展方向

- 增加红绿灯识别与交通标志处理。
- 增加曲率预测，提高高速巡线稳定性。
- 将视觉模块和底盘控制整合到同一硬件平台。
- 增加避障、路口任务和路径规划功能。
